## yw]op

In [2]:
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 MB 406.7 kB/s eta 0:00:00m eta 0:00:010:00:05
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 376.9 kB/s eta 0:00:00 kB/s eta 0:00:01:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 304.9 kB/s eta 0:00:00m eta 0:00:010:00:03

[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import fitz 
import json
import re

/Users/anhleduc/workspace/rag-finvisor-vn/data-pipeline/.venv/lib/python3.13/site-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [4]:
def extract_text_from_structured_pdf(pdf_file):
    """
    Extracts text from a structured PDF file page by page,
    preserving text layout block structures.
    """
    doc = fitz.open(pdf_file)
    extracted_data = []
    
    print(f"Total pages in PDF: {len(doc)}")
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        
        # Extract text by blocks to preserve paragraphs and headings structure
        # Each block tuple: (x0, y0, x1, y1, "text", block_no, block_type)
        blocks = page.get_text("blocks")
        
        page_content = []
        for b in blocks:
            text = b[4].strip()
            if text:  # Skip empty text blocks
                page_content.append({
                    "box": [round(b[0], 2), round(b[1], 2), round(b[2], 2), round(b[3], 2)],
                    "text": text
                })
        
        extracted_data.append({
            "page": page_num + 1,
            "content": page_content
        })
        
    return extracted_data



In [11]:
pdf_path = "/Users/anhleduc/workspace/rag-finvisor-vn/pack_vn/tmp/bao-cao-tai-chinh-hop-nhat-quy-2-2024-dlpn.pdf"

In [12]:
# Run text extraction function
pdf_data = extract_text_from_structured_pdf(pdf_path)

Total pages in PDF: 6


In [13]:
pdf_data

[{'page': 1,
  'content': [{'box': [51.02, 31.23, 555.62, 38.63],
    'text': 'Điện lực Phương Nam\nBáo cáo tài chính hợp nhất quý 2 năm 2024'},
   {'box': [51.02, 807.11, 555.59, 813.71],
    'text': 'Tài liệu mô phỏng phục vụ đào tạo\nTrang 1'},
   {'box': [120.84, 122.99, 485.76, 137.99],
    'text': 'CÔNG TY CỔ PHẦN ĐIỆN LỰC PHƯƠNG NAM'},
   {'box': [151.3, 149.41, 455.37, 174.91],
    'text': 'Mã chứng khoán: DLPN   ·   Sàn niêm yết: HOSE\nSố 72 Võ Văn Kiệt, phường Cầu Ông Lãnh, TP Hồ Chí Minh'},
   {'box': [98.06, 220.18, 508.56, 235.18],
    'text': 'BÁO CÁO TÀI CHÍNH HỢP NHẤT QUÝ 2 NĂM 2024'},
   {'box': [194.02, 243.77, 412.6, 254.27],
    'text': 'Cho kỳ kế toán kết thúc ngày 30/06/2024'},
   {'box': [243.25, 270.11, 363.38, 280.61], 'text': '(Chưa được kiểm toán)'},
   {'box': [244.08, 330.46, 362.56, 340.96],
    'text': 'Đơn vị tính: triệu đồng'},
   {'box': [58.65, 418.46, 548.07, 426.06],
    'text': 'TÀI LIỆU MÔ PHỎNG PHỤC VỤ ĐÀO TẠO — số liệu được sinh tự động, doanh n

In [ ]:
# Print extracted text blocks for the first two pages
for page_info in pdf_data[:2]:
    print(f"--- PAGE {page_info['page']} ---")
    for block in page_info['content']:
        print(block['text'])
        print("-" * 30)

In [15]:
def process_to_rag_corpus(raw_pages, output_filename="corpus_output.jsonl"):
    """
    Processes raw extracted pages into structured chunks with metadata 
    and exports them in JSONL format for vector databases.
    """
    corpus_records = []
    
    for idx, page_item in enumerate(raw_pages):
        page_no = page_item["page"]
        content = page_item["content"]
        
        # Determine section name based on page or content keywords (customizable logic)
        section_name = "THÔNG TIN CHUNG"
        if "BẢNG CÂN ĐỐI KẾ TOÁN" in content:
            section_name = "BẢNG CÂN ĐỐI KẾ TOÁN"
        elif "BÁO CÁO KẾT QUẢ HOẠT ĐỘNG KINH DOANH" in content:
            section_name = "BÁO CÁO KẾT QUẢ HOẠT ĐỘNG KINH DOANH"
        elif "BÁO CÁO LƯU CHUYỂN TIỀN TỆ" in content:
            section_name = "BÁO CÁO LƯU CHUYỂN TIỀN TỆ"
        elif "THUYẾT MINH" in content or "V.0" in content:
            section_name = "BẢN THUYẾT MINH BÁO CÁO TÀI CHÍNH"

        # Create structured record
        record = {
            "id": f"chunk_{idx+1:03d}",
            "text": content,
            "metadata": {
                "company": "Công ty Cổ phần Điện lực Phương Nam",
                "ticker": "DLPN",
                "year": 2024,
                "period": "Quý 2",
                "page": page_no,
                "section": section_name,
                "doc_type": "Báo cáo tài chính hợp nhất"
            }
        }
        corpus_records.append(record)
        
    # Save to JSONL file
    with open(output_filename, "w", encoding="utf-8") as f:
        for rec in corpus_records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            
    print(f"Successfully generated corpus file: {output_filename} with {len(corpus_records)} chunks!")
    return corpus_records



In [17]:
# Run processing function
structured_corpus = process_to_rag_corpus(pdf_data, output_filename="rag_corpus_output.jsonl")

Successfully generated corpus file: rag_corpus_output.jsonl with 6 chunks!


In [18]:
# Install Docling library
!pip install docling

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 226.9 kB/s eta 0:00:001m229.0 kB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 315.1 kB/s eta 0:00:001m308.8 kB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 280.8 kB/s eta 0:00:00 kB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 280.3 kB/s eta 0:00:001m280.6 kB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 236.2 kB/s eta 0:00:00 kB/s eta 0:00:01:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 363.2 kB/s eta 0:00:00 kB/s eta 0:00:01:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 318.8 kB/s eta 0:00:00m eta 0:00:010:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 381.8 kB/s eta 0:00:000:00:01m eta 0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.3/127.3 MB 300.8 kB/s eta 0:00:00m eta 0:00:010:00:11
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 338.5 kB/s eta 0:00:001m343.4 kB/s eta 0:00:

In [ ]:
from docling.document_converter import DocumentConverter

# Initialize Document Converter
converter = DocumentConverter()

# Convert the target PDF file
pdf_file = "PLX_BCTC_HN_N2025.pdf"
result = converter.convert(pdf_file)

# Export document content to Markdown format
markdown_output = result.document.export_to_markdown()

# Save the Markdown output for RAG chunking pipeline
output_md_path = "plx_bctc_parsed.md"
with open(output_md_path, "w", encoding="utf-8") as f:
    f.write(markdown_output)

print(f"Successfully converted the entire PDF into structured Markdown: {output_md_path}")

In [30]:
!pip install paddlepaddle paddleocr

  Using cached pillow-13.0.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (9.1 kB)
  Using cached safetensors-0.8.0-cp310-abi3-macosx_11_0_arm64.whl.metadata (4.2 kB)
  Using cached setuptools-84.2.1-py3-none-any.whl.metadata (6.6 kB)
  Using cached colorlog-6.12.0-py3-none-any.whl.metadata (11 kB)
  Using cached filelock-4.2.4-py3-none-any.whl.metadata (2.0 kB)
  Using cached pydantic-2.14.1-py3-none-any.whl.metadata (133 kB)
  Using cached pyclipper-1.4.0-cp313-cp313-macosx_10_13_universal2.whl.metadata (8.6 kB)
  Using cached pypdfium2-5.14.0-py3-none-macosx_13_0_arm64.whl.metadata (62 kB)
  Using cached shapely-2.2.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (6.7 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.50.1-cp313-cp313-macosx_11_0_arm64.whl.metadata (6.5 kB)
  Using cached typing_inspection-0.4.4-

In [ ]:
import fitz  # PyMuPDF for structured PDFs
import json
import numpy as np
from pathlib import Path  # For recursive directory scanning
from paddleocr import PaddleOCR  # PaddleOCR for scanned PDFs/images

# ==========================================
# Step 0: Initialize PaddleOCR Engine (Optimized)
# ==========================================
ocr_engine = PaddleOCR(use_textline_orientation=False, lang='vi')

# ==========================================
# Step 1: Recursively find all PDF files
# ==========================================
def get_all_pdf_files(folder_path):
    """
    Recursively scans the folder and subfolders to find all PDF files.
    """
    path = Path(folder_path)
    pdf_files = list(path.rglob("*.pdf"))
    return pdf_files


# ==========================================
# Step 2: Hybrid Extraction (PyMuPDF + PaddleOCR Fallback)
# ==========================================
def extract_text_hybrid(pdf_path):
    """
    Extracts text page by page. Uses PyMuPDF first; if text is empty (scanned PDF), 
    it falls back to PaddleOCR.
    """
    try:
        doc = fitz.open(pdf_path)
    except Exception as e:
        print(f"Error opening file {pdf_path}: {e}")
        return []
        
    extracted_pages = []
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        
        # 1. Try extracting standard text blocks using PyMuPDF
        blocks = page.get_text("blocks")
        page_content = []
        full_page_text = ""
        
        for b in blocks:
            text = b[4].strip()
            if text:
                page_content.append({
                    "box": [round(b[0], 2), round(b[1], 2), round(b[2], 2), round(b[3], 2)],
                    "text": text
                })
                full_page_text += text + "\n"
        
        # 2. Fallback to PaddleOCR if the page has little to no text (e.g., scanned PDF/image)
        if len(full_page_text.strip()) < 50:
            print(f"  -> Page {page_num + 1} seems to be scanned. Running PaddleOCR...")
            
            # Render PDF page to image (DPI 200 for better OCR accuracy)
            pix = page.get_pixmap(dpi=200)
            img_np = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
            if pix.n == 4:  # Convert RGBA to RGB
                img_np = img_np[:, :, :3]
                
            # Run PaddleOCR using the updated .predict() API
            ocr_results = ocr_engine.predict(img_np)
            
            page_content = []
            full_page_text = ""
            
            if ocr_results:
                for res in ocr_results:
                    # Safely extract dictionary representation from PaddleX result object (fixed json property)
                    res_dict = None
                    if hasattr(res, 'json'):
                        res_dict = res.json if isinstance(res.json, dict) else res.json() if callable(res.json) else None
                    elif isinstance(res, dict):
                        res_dict = res
                    elif hasattr(res, 'to_dict'):
                        res_dict = res.to_dict()
                    
                    boxes, texts, scores = None, None, None
                    
                    if res_dict:
                        boxes = res_dict.get('dt_polys') or res_dict.get('boxes') or res_dict.get('boxes_ocr')
                        texts = res_dict.get('rec_texts') or res_dict.get('texts')
                        scores = res_dict.get('rec_scores') or res_dict.get('scores')
                    else:
                        boxes = getattr(res, 'dt_polys', None) or getattr(res, 'boxes', None)
                        texts = getattr(res, 'rec_texts', None) or getattr(res, 'texts', None)
                        scores = getattr(res, 'rec_scores', None) or getattr(res, 'scores', None)

                    if boxes is not None and texts is not None:
                        if scores is None:
                            scores = [1.0] * len(texts)
                        for box_coords, text, score in zip(boxes, texts, scores):
                            text_str = str(text).strip()
                            if text_str:
                                if len(box_coords) >= 4:
                                    flat_box = [
                                        round(float(box_coords[0][0]), 2),
                                        round(float(box_coords[0][1]), 2),
                                        round(float(box_coords[2][0]), 2),
                                        round(float(box_coords[2][1]), 2)
                                    ]
                                else:
                                    flat_box = [0.0, 0.0, 0.0, 0.0]
                                
                                page_content.append({
                                    "box": flat_box,
                                    "text": text_str,
                                    "confidence": round(float(score), 2)
                                })
                                full_page_text += text_str + "\n"
                        
        extracted_pages.append({
            "page": page_num + 1,
            "full_text": full_page_text.strip(),
            "content": page_content
        })
        
    return extracted_pages


# ==========================================
# Step 3: Build unified JSON Corpus for RAG
# ==========================================
def build_corpus_from_directory(input_folder, output_json_file="corpus_output.json"):
    pdf_files = get_all_pdf_files(input_folder)
    print(f"Found a total of {len(pdf_files)} PDF files. Processing with Hybrid OCR/Text pipeline...")
    
    corpus_records = []
    global_chunk_id = 1
    
    for pdf_path in pdf_files:
        print(f"\n--> Processing file: {pdf_path.name}")
        pages_data = extract_text_hybrid(pdf_path)
        
        for page_data in pages_data:
            page_no = page_data["page"]
            full_text = page_data["full_text"]
            content_blocks = page_data["content"]
            
            # Automatically classify financial sections based on keywords
            section_name = "GENERAL INFORMATION"
            if "BẢNG CÂN ĐỐI KẾ TOÁN" in full_text:
                section_name = "STATEMENT OF FINANCIAL POSITION"
            elif "BÁO CÁO KẾT QUẢ HOẠT ĐỘNG KINH DOANH" in full_text:
                section_name = "INCOME STATEMENT"
            elif "BÁO CÁO LƯU CHUYỂN TIỀN TỆ" in full_text:
                section_name = "CASH FLOW STATEMENT"
            elif "THUYẾT MINH" in full_text or "BẢN THUYẾT MINH" in full_text:
                section_name = "NOTES TO THE FINANCIAL STATEMENTS"
            
            # Create standardized RAG corpus record
            record = {
                "id": f"chunk_{global_chunk_id:05d}",
                "text": full_text,
                "content_blocks": content_blocks,
                "metadata": {
                    "source_file": pdf_path.name,
                    "file_path": str(pdf_path),
                    "page": page_no,
                    "section": section_name,
                    "doc_type": "Financial Report (Hybrid OCR)"
                }
            }
            corpus_records.append(record)
            global_chunk_id += 1
            
    # Save output to JSON file
    with open(output_json_file, "w", encoding="utf-8") as f:
        json.dump(corpus_records, f, ensure_ascii=False, indent=4)
        
    print(f"\n==========================================")
    print(f"Completed! Generated {len(corpus_records)} chunks.")
    print(f"Corpus saved to: {output_json_file}")
    return corpus_records


# ==========================================
# Execution
# ==========================================
if __name__ == "__main__":
    input_dir = "/Users/anhleduc/workspace/rag-finvisor-vn/pack_vn/corpus"
    output_file = "/Users/anhleduc/workspace/rag-finvisor-vn/pack_vn/corpus_master_output_with_ocr.json"
    
    corpus_data = build_corpus_from_directory(input_dir, output_file)

Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/anhleduc/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/anhleduc/.paddlex/official_models/UVDoc`.
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/anhleduc/.paddlex/official_models/PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/anhleduc/.paddlex/official_models/PP-OCRv6_medium_rec`.


Found a total of 46 PDF files. Processing with Hybrid OCR/Text pipeline...

--> Processing file: PLX_BCTC_HN_N2025.pdf
  -> Page 2 seems to be scanned. Running PaddleOCR...
  -> Page 3 seems to be scanned. Running PaddleOCR...
  -> Page 4 seems to be scanned. Running PaddleOCR...
  -> Page 5 seems to be scanned. Running PaddleOCR...
  -> Page 6 seems to be scanned. Running PaddleOCR...
  -> Page 7 seems to be scanned. Running PaddleOCR...
  -> Page 8 seems to be scanned. Running PaddleOCR...
  -> Page 9 seems to be scanned. Running PaddleOCR...
  -> Page 10 seems to be scanned. Running PaddleOCR...
  -> Page 11 seems to be scanned. Running PaddleOCR...
  -> Page 12 seems to be scanned. Running PaddleOCR...
  -> Page 13 seems to be scanned. Running PaddleOCR...
  -> Page 14 seems to be scanned. Running PaddleOCR...
  -> Page 15 seems to be scanned. Running PaddleOCR...
  -> Page 16 seems to be scanned. Running PaddleOCR...
  -> Page 17 seems to be scanned. Running PaddleOCR...
  -> Page

In [ ]:
# =========================================================
# THỰC THI CHƯƠNG TRÌNH VỚI ĐƯỜNG DẪN CỦA BẠN
# =========================================================
input_dir = "/Users/anhleduc/workspace/rag-finvisor-vn/pack_vn/corpus"
output_file = "/Users/anhleduc/workspace/rag-finvisor-vn/pack_vn/corpus_master_output.json"

corpus_data = build_corpus_from_directory(input_dir, output_file)